
# Scala Functional Data Structures — Deep Dive

During this lecture, we will cover:

- Algebraic Data Types (ADTs)
- Sum Types and Product Types
- Sealed Traits and Case Classes
- Pattern Matching (Basics)
- Guards, Nested Patterns, and Destructuring
- Higher-Order Functions on Collections
- `map`, `filter`, `flatMap`, `foldLeft`, `foldRight`, `reduce`, `scan`
- Closures and Captured Variables
- Scopes: Lexical Scope, Block Scope, and Closure Scope
- Combining All Techniques — Real-World Pipelines

These concepts form the **foundation of idiomatic Scala** and appear constantly in **data engineering, domain modeling, stream processing, and large-scale Scala systems** such as Apache Spark, Akka, and Cats.

Mastering them will allow you to model complex domains precisely, process data expressively, and write code that is **safe, readable, and composable**.



# 1. Algebraic Data Types — Introduction

An **Algebraic Data Type (ADT)** is a type formed by **combining other types** using two fundamental operations:

- **Sum types** (also called *union types* or *coproducts*): a value is **one of** several alternatives
- **Product types** (also called *record types* or *tuples*): a value **contains** several values simultaneously

The name "algebraic" comes from abstract algebra:

| Algebra | Type Theory | Scala |
|---------|------------|-------|
| Sum (addition) | Sum type | `sealed trait` with alternatives |
| Product (multiplication) | Product type | `case class` |

ADTs are the primary tool for **domain modeling** in functional programming. They allow you to make illegal states unrepresentable at the type level.

The classic slogan: **"Make illegal states unrepresentable."**



# 2. Product Types — Case Classes

A **product type** holds multiple values simultaneously. In Scala, `case class` is the idiomatic way to define product types.

The name "product" refers to the fact that the number of possible values of the type is the **product** of the possible values of each field.

For example, `case class Point(x: Int, y: Int)` can represent `Int.MaxValue * Int.MaxValue` combinations — the product of the two fields.

Case classes come with:
- Immutable fields by default
- Auto-generated `equals`, `hashCode`, and `toString`
- A `copy` method for creating modified instances
- Pattern matching support


In [ ]:
// A simple product type
case class Point(x: Double, y: Double)

// A richer product type
case class Person(name: String, age: Int, email: String)

// Creating instances
val p1 = Point(3.0, 4.0)
val alice = Person("Alice", 30, "alice@example.com")

// copy creates a modified version without mutating the original
val olderAlice = alice.copy(age = 31)

// Structural equality is built in
val p2 = Point(3.0, 4.0)
p1 == p2       // true — structural equality

// toString is auto-generated
alice.toString


### Nested Product Types

Product types can contain other product types, forming rich nested structures.


In [ ]:
case class Address(street: String, city: String, country: String)
case class Company(name: String, headOffice: Address)
case class Employee(id: Int, person: Person, company: Company, role: String)

val emp = Employee(
  id = 1,
  person = Person("Bob", 28, "bob@corp.com"),
  company = Company("Acme", Address("123 Main St", "Istanbul", "Turkey")),
  role = "Engineer"
)

// Accessing nested fields
emp.person.name
emp.company.headOffice.city

// Deep copy — update a nested field
val movedEmp = emp.copy(
  company = emp.company.copy(
    headOffice = emp.company.headOffice.copy(city = "Ankara")
  )
)

movedEmp.company.headOffice.city


The `copy` method is the standard way to perform **immutable updates** on deeply nested structures. Libraries like Monocle provide lenses for more ergonomic deep updates.



# 3. Sum Types — Sealed Traits

A **sum type** represents a value that is exactly **one of** several alternatives.

The name "sum" comes from the fact that the total number of possible values is the **sum** of the values in each alternative.

In Scala, sum types are defined using a `sealed trait` (or `sealed abstract class`) combined with `case class` and `case object` variants.

The `sealed` keyword is critical — it tells the compiler that **all subclasses are defined in the same file**, enabling **exhaustiveness checking** in pattern matching.


In [ ]:
// A sum type representing a traffic light
sealed trait TrafficLight
case object Red    extends TrafficLight
case object Yellow extends TrafficLight
case object Green  extends TrafficLight

// A sum type representing the result of a computation
sealed trait Result[+A]
case class  Success[A](value: A) extends Result[A]
case class  Failure(error: String) extends Result[Nothing]

// Using them
val light: TrafficLight = Red
val ok: Result[Int]     = Success(42)
val err: Result[Int]    = Failure("Division by zero")

light
ok
err


### Why `sealed`?

The `sealed` modifier forces all variants to be defined in the same file. This gives the compiler full knowledge of all possible values, enabling it to warn when a pattern match is **non-exhaustive** — i.e., when you haven't handled all possible cases.

Without `sealed`, any file could add a new subtype, making pattern matching on the trait dangerous at runtime.


In [ ]:
// The compiler warns if you don't cover all cases
sealed trait Shape
case class Circle(radius: Double)           extends Shape
case class Rectangle(width: Double, height: Double) extends Shape
case class Triangle(base: Double, height: Double)   extends Shape

def area(shape: Shape): Double = shape match {
  case Circle(r)          => Math.PI * r * r
  case Rectangle(w, h)    => w * h
  case Triangle(b, h)     => 0.5 * b * h
  // If you remove Triangle case, the compiler will warn: match may not be exhaustive
}

area(Circle(5.0))
area(Rectangle(4.0, 6.0))
area(Triangle(3.0, 8.0))


# 4. Combining Sum and Product Types — Recursive ADTs

The real power of ADTs comes from **combining sum and product types** to model complex domains.

A classic example is a **binary tree** — a recursive data structure that is either:
- a `Leaf` (empty node), or
- a `Node` containing a value, a left subtree, and a right subtree

This definition is naturally expressed as an ADT.


In [ ]:
sealed trait Tree[+A]
case object Leaf extends Tree[Nothing]
case class  Node[A](value: A, left: Tree[A], right: Tree[A]) extends Tree[A]

// Build a small tree
val tree: Tree[Int] =
  Node(1,
    Node(2,
      Node(4, Leaf, Leaf),
      Node(5, Leaf, Leaf)
    ),
    Node(3,
      Node(6, Leaf, Leaf),
      Leaf
    )
  )

// Recursive function using pattern matching
def depth[A](tree: Tree[A]): Int = tree match {
  case Leaf           => 0
  case Node(_, l, r)  => 1 + Math.max(depth(l), depth(r))
}

def size[A](tree: Tree[A]): Int = tree match {
  case Leaf           => 0
  case Node(_, l, r)  => 1 + size(l) + size(r)
}

depth(tree)
size(tree)


### A Linked List as an ADT

Scala's own `List[A]` is itself an ADT. We can model our own linked list to understand the structure.


In [ ]:
sealed trait MyList[+A]
case object Nil extends MyList[Nothing]
case class  Cons[A](head: A, tail: MyList[A]) extends MyList[A]

// Build a list: 1 :: 2 :: 3 :: Nil
val myList: MyList[Int] = Cons(1, Cons(2, Cons(3, Nil)))

def myLength[A](list: MyList[A]): Int = list match {
  case Nil        => 0
  case Cons(_, t) => 1 + myLength(t)
}

def mySum(list: MyList[Int]): Int = list match {
  case Nil         => 0
  case Cons(h, t)  => h + mySum(t)
}

def myMap[A, B](list: MyList[A])(f: A => B): MyList[B] = list match {
  case Nil         => Nil
  case Cons(h, t)  => Cons(f(h), myMap(t)(f))
}

myLength(myList)
mySum(myList)
myMap(myList)(_ * 10)


This mirrors exactly how Scala's standard `List[A]` works internally — as a recursive ADT of `::` (cons) and `Nil`.



# 5. Modeling a Domain with ADTs

Let's model a real-world domain — **payment processing** — entirely using ADTs.

The goal is to make every possible state of a payment explicit at the type level, so invalid transitions (like refunding a failed payment) become compile-time errors.


In [ ]:
// Payment method — a sum type
sealed trait PaymentMethod
case class CreditCard(number: String, expiry: String, cvv: String) extends PaymentMethod
case class BankTransfer(iban: String, bic: String)                 extends PaymentMethod
case object CryptoWallet                                           extends PaymentMethod

// Payment status — a sum type
sealed trait PaymentStatus
case object Pending                           extends PaymentStatus
case class  Authorized(authCode: String)      extends PaymentStatus
case class  Captured(transactionId: String)   extends PaymentStatus
case class  Refunded(reason: String)          extends PaymentStatus
case class  Failed(errorCode: Int, msg: String) extends PaymentStatus

// Payment — a product type combining the above
case class Payment(
  id: String,
  amount: BigDecimal,
  currency: String,
  method: PaymentMethod,
  status: PaymentStatus
)

// Create a payment
val payment = Payment(
  id = "PAY-001",
  amount = BigDecimal("99.99"),
  currency = "USD",
  method = CreditCard("4111111111111111", "12/27", "123"),
  status = Pending
)

// Transition: authorize
val authorized = payment.copy(status = Authorized("AUTH-XYZ"))
// Transition: capture
val captured = authorized.copy(status = Captured("TXN-456"))

captured


This model is **self-documenting**: the type system expresses all possible payment states. Any function operating on a `Payment` must handle every possible `PaymentStatus` — the compiler enforces completeness.



# 6. Pattern Matching — Basics

**Pattern matching** is the primary way to **inspect and deconstruct** ADTs in Scala.

It is more powerful than a traditional `switch` statement — it can:

- match on type
- destructure nested structures
- bind matched components to names
- apply guards (conditional filters)
- match on literals, tuples, and wildcards

The syntax is `expression match { case pattern => result ... }`


In [ ]:
// Matching on literals
def dayName(n: Int): String = n match {
  case 1 => "Monday"
  case 2 => "Tuesday"
  case 3 => "Wednesday"
  case 4 => "Thursday"
  case 5 => "Friday"
  case 6 => "Saturday"
  case 7 => "Sunday"
  case _ => "Unknown"
}

dayName(3)
dayName(7)
dayName(9)


The `_` wildcard is a catch-all pattern — it matches anything and is equivalent to the `default` case in a switch statement.



### Matching on Case Classes

Case classes support **destructuring** — binding their fields to names in one operation.


In [ ]:
case class Point(x: Double, y: Double)

def describePoint(p: Point): String = p match {
  case Point(0, 0)   => "Origin"
  case Point(x, 0)   => s"On the X-axis at $x"
  case Point(0, y)   => s"On the Y-axis at $y"
  case Point(x, y)   => s"At ($x, $y)"
}

describePoint(Point(0, 0))
describePoint(Point(5, 0))
describePoint(Point(0, 3))
describePoint(Point(2, 4))


### Matching on Sealed Traits

Pattern matching on sealed traits produces **exhaustiveness-checked code**.


In [ ]:
sealed trait Expr
case class Num(value: Double)           extends Expr
case class Add(left: Expr, right: Expr) extends Expr
case class Mul(left: Expr, right: Expr) extends Expr
case class Neg(expr: Expr)              extends Expr

def eval(expr: Expr): Double = expr match {
  case Num(v)      => v
  case Add(l, r)   => eval(l) + eval(r)
  case Mul(l, r)   => eval(l) * eval(r)
  case Neg(e)      => -eval(e)
}

// Represents: (3 + 4) * -(2)
val expression = Mul(Add(Num(3), Num(4)), Neg(Num(2)))
eval(expression)


We just built a small **expression evaluator** entirely through pattern matching on an ADT. This is the foundation of interpreter design, compiler front-ends, and rule engines.



# 7. Pattern Matching — Guards

A **guard** adds a conditional `if` expression to a pattern case. The case only matches when both the pattern and the guard are satisfied.

Guards allow fine-grained control without nesting `if-else` inside pattern branches.


In [ ]:
def classifyNumber(n: Int): String = n match {
  case 0                   => "Zero"
  case x if x < 0         => s"Negative: $x"
  case x if x % 2 == 0    => s"Positive even: $x"
  case x                   => s"Positive odd: $x"
}

classifyNumber(0)
classifyNumber(-7)
classifyNumber(8)
classifyNumber(13)


### Guards on Case Classes


In [ ]:
case class Order(id: String, total: Double, isPriority: Boolean)

def shippingCost(order: Order): Double = order match {
  case Order(_, _, true)                  => 0.0           // priority always free
  case Order(_, total, _) if total > 100  => 0.0           // free over $100
  case Order(_, total, _) if total > 50   => 4.99          // reduced
  case _                                  => 9.99          // standard
}

shippingCost(Order("A1", 25.0, true))
shippingCost(Order("A2", 150.0, false))
shippingCost(Order("A3", 75.0, false))
shippingCost(Order("A4", 20.0, false))


# 8. Pattern Matching — Nested and Tuple Patterns

Pattern matching can be applied **recursively** to destructure nested structures.


In [ ]:
// Matching on Option
def describeOption(opt: Option[Int]): String = opt match {
  case None          => "Nothing here"
  case Some(0)       => "Got zero"
  case Some(n) if n < 0 => s"Got negative: $n"
  case Some(n)       => s"Got positive: $n"
}

describeOption(None)
describeOption(Some(0))
describeOption(Some(-5))
describeOption(Some(42))

In [ ]:
// Matching on tuples
def classifyPair(pair: (String, Int)): String = pair match {
  case (name, age) if age < 18  => s"$name is a minor"
  case (name, age) if age >= 65 => s"$name is a senior"
  case (name, _)                => s"$name is an adult"
}

classifyPair(("Alice", 30))
classifyPair(("Tommy", 14))
classifyPair(("Grace", 70))

In [ ]:
// Matching on nested case classes
case class Address(city: String, country: String)
case class User(name: String, address: Address)

def greet(user: User): String = user match {
  case User(name, Address(_, "Turkey"))    => s"Merhaba, $name!"
  case User(name, Address(_, "Germany"))   => s"Hallo, $name!"
  case User(name, Address(city, country))  => s"Hello, $name from $city, $country!"
}

greet(User("Mehmet", Address("Istanbul", "Turkey")))
greet(User("Hans", Address("Berlin", "Germany")))
greet(User("Emma", Address("London", "UK")))


### Matching on Lists

Lists have a special pattern syntax: `head :: tail` deconstructs a list into its first element and the rest.


In [ ]:
def describeList[A](list: List[A]): String = list match {
  case Nil             => "Empty list"
  case x :: Nil        => s"Single element: $x"
  case x :: y :: Nil   => s"Two elements: $x and $y"
  case x :: rest       => s"Starts with $x, then ${rest.length} more elements"
}

describeList(List[Int]())
describeList(List(42))
describeList(List(1, 2))
describeList(List(1, 2, 3, 4, 5))

In [ ]:
// Recursive algorithm using list patterns
def myReverse[A](list: List[A]): List[A] = list match {
  case Nil       => Nil
  case h :: t    => myReverse(t) :+ h
}

def sumFirst3(list: List[Int]): Int = list match {
  case a :: b :: c :: _ => a + b + c
  case a :: b :: Nil    => a + b
  case a :: Nil         => a
  case Nil              => 0
}

myReverse(List(1, 2, 3, 4, 5))
sumFirst3(List(10, 20, 30, 40, 50))
sumFirst3(List(5, 6))


# 9. Pattern Matching — Type Patterns and Binding

Pattern matching can also **dispatch on the runtime type** of a value, and you can use the `@` operator to **bind** the matched value to a name.


In [ ]:
// Type-based dispatch
def describe(value: Any): String = value match {
  case i: Int     => s"Integer: $i"
  case s: String  => s"String of length ${s.length}: '$s'"
  case d: Double  => s"Double: $d"
  case b: Boolean => s"Boolean: $b"
  case _          => s"Unknown type: ${value.getClass.getSimpleName}"
}

describe(42)
describe("hello")
describe(3.14)
describe(true)

In [ ]:
// @ binding: bind the whole matched object AND destructure it
case class Rectangle(w: Double, h: Double)

def describeRect(r: Rectangle): String = r match {
  case rect @ Rectangle(w, h) if w == h =>
    s"$rect is a square with side $w"
  case rect @ Rectangle(w, h) =>
    s"$rect is a rectangle ${w}x${h}, area = ${w * h}"
}

describeRect(Rectangle(5.0, 5.0))
describeRect(Rectangle(3.0, 7.0))


### Pattern Matching in `val` Declarations

Pattern matching is not limited to `match` expressions — it can also appear in `val` declarations for **destructuring assignment**.


In [ ]:
// Destructuring a tuple
val (first, second, third) = (10, "hello", true)
first
second

// Destructuring a case class
case class Config(host: String, port: Int, debug: Boolean)
val Config(h, p, d) = Config("localhost", 8080, false)
h
p

// Destructuring the head of a list
val head :: tail = List(1, 2, 3, 4, 5)
head
tail


Destructuring in `val` declarations is a powerful shorthand that makes code more readable when you know the structure of the value at compile time.



# 10. Higher-Order Functions on Collections — Overview

A **higher-order function (HOF)** is a function that takes other functions as parameters or returns functions.

Scala's standard library collections (`List`, `Vector`, `Set`, `Map`, `Option`, etc.) are built around a rich set of HOFs.

These functions implement the most common data transformation patterns:

| Function | What it does |
|----------|--------------|
| `map` | Transform every element |
| `filter` | Keep elements matching a predicate |
| `flatMap` | Transform and flatten |
| `foldLeft` | Accumulate from left to right |
| `foldRight` | Accumulate from right to left |
| `reduce` | Combine all elements |
| `scan` | Running fold — keep all intermediate results |
| `groupBy` | Partition into a `Map` by key function |
| `zip` | Combine two collections pairwise |
| `collect` | Filter and transform using a `PartialFunction` |
| `partition` | Split into two lists by predicate |
| `span` | Split at first element failing predicate |

Together these functions replace most imperative loops and make collection processing **declarative, composable, and concise**.



# 11. `map` — Transform Every Element

`map` applies a function to **every element** of a collection and returns a new collection of the same length containing the results.

Signature: `List[A].map(f: A => B): List[B]`

`map` never changes the **structure** of the collection — only the **values** inside it.


In [ ]:
val numbers = List(1, 2, 3, 4, 5)

// Transform integers
numbers.map(n => n * n)          // square each
numbers.map(n => n.toDouble)     // convert to Double
numbers.map(n => s"Item $n")     // convert to String

// Map over a list of strings
val names = List("alice", "bob", "charlie")
names.map(_.capitalize)
names.map(_.toUpperCase)
names.map(n => (n, n.length))    // pair each name with its length

In [ ]:
// map over Option — applies f only if value is present
val maybeNum: Option[Int] = Some(10)
val noNum: Option[Int] = None

maybeNum.map(_ * 2)   // Some(20)
noNum.map(_ * 2)      // None

// Chaining maps
val result = maybeNum
  .map(_ + 5)        // Some(15)
  .map(_ * 3)        // Some(45)
  .map(n => s"Result: $n")

result


### `map` on Case Classes


In [ ]:
case class Product(name: String, price: Double, category: String)

val products = List(
  Product("Laptop",  999.99, "Electronics"),
  Product("Desk",    299.50, "Furniture"),
  Product("Monitor", 349.00, "Electronics")
)

// Apply a 10% discount to all products
val discounted = products.map(p => p.copy(price = p.price * 0.9))

// Extract just names
val names = products.map(_.name)

// Add VAT
val withVat = products.map(p => p.copy(price = p.price * 1.2))

discounted
names
withVat


# 12. `filter` — Keep Elements Matching a Predicate

`filter` returns a new collection containing only those elements for which the predicate function returns `true`.

Signature: `List[A].filter(p: A => Boolean): List[A]`

`filter` preserves the **type** of the collection but may change its **length**.


In [ ]:
val numbers = List(1, -2, 3, -4, 5, -6, 7, -8, 9, 10)

numbers.filter(_ > 0)         // keep positives
numbers.filter(_ % 2 == 0)    // keep evens
numbers.filterNot(_ < 0)      // remove negatives (filterNot = filter with negated predicate)

val words = List("scala", "java", "python", "haskell", "rust")
words.filter(_.length > 4)
words.filter(_.startsWith("s"))

In [ ]:
// Combining filter and map — the classic pipeline
case class Student(name: String, grade: Double, passed: Boolean)

val students = List(
  Student("Alice",   92.0, true),
  Student("Bob",     45.0, false),
  Student("Charlie", 78.5, true),
  Student("Diana",   33.0, false),
  Student("Eve",     88.0, true)
)

// Get names of passing students with grade >= 80
val honorRoll = students
  .filter(_.passed)
  .filter(_.grade >= 80)
  .map(_.name)

honorRoll


### `partition` — Split in Two

`partition` is like running `filter` and `filterNot` at the same time — it returns a pair of lists.


In [ ]:
val nums = List(1, 2, 3, 4, 5, 6, 7, 8, 9, 10)
val (evens, odds) = nums.partition(_ % 2 == 0)

evens
odds

// Partition students into pass/fail
val (passed, failed) = students.partition(_.passed)
passed.map(_.name)
failed.map(_.name)


# 13. `flatMap` — Transform and Flatten

`flatMap` applies a function that returns a collection to each element, then **flattens** the result into a single collection.

Signature: `List[A].flatMap(f: A => List[B]): List[B]`

It is equivalent to `map` followed by `flatten`.

`flatMap` is the key operation for working with collections of collections — and for **monadic composition**.


In [ ]:
val sentences = List("hello world", "scala is great", "functional programming")

// Split each sentence into words — flatMap flattens the list of lists
val words = sentences.flatMap(_.split(" ").toList)
words

// Compare with map (produces nested lists)
val nested = sentences.map(_.split(" ").toList)
nested

In [ ]:
// flatMap on Option — chain optional computations
def safeDivide(a: Int, b: Int): Option[Int] =
  if (b == 0) None else Some(a / b)

def safeSqrt(n: Int): Option[Double] =
  if (n < 0) None else Some(Math.sqrt(n))

// Chain optional computations
Some(16)
  .flatMap(n => safeDivide(100, n))
  .flatMap(n => safeSqrt(n * 4))

In [ ]:
// flatMap for generating combinations
val sizes  = List("S", "M", "L")
val colors = List("Red", "Blue", "Green")

// All size-color combinations
val combinations = sizes.flatMap(s => colors.map(c => s"$s-$c"))
combinations


The combination generation pattern with `flatMap` is equivalent to a **cartesian product** and is the same as a `for`-comprehension with two generators.



# 14. `foldLeft` — Accumulate from Left to Right

`foldLeft` is the most powerful and general HOF on collections. Almost every other HOF can be implemented in terms of `foldLeft`.

It processes a collection from **left to right**, carrying an **accumulator** that is updated at each step.

Signature: `List[A].foldLeft(z: B)(f: (B, A) => B): B`

- `z` is the **initial value** (zero/identity element)
- `f` is the **combining function** that takes the current accumulator and the next element


In [ ]:
val nums = List(1, 2, 3, 4, 5)

// Sum
nums.foldLeft(0)(_ + _)

// Product
nums.foldLeft(1)(_ * _)

// Maximum
nums.foldLeft(Int.MinValue)(Math.max)

// Build a reversed list
nums.foldLeft(List.empty[Int])((acc, x) => x :: acc)

// Count elements matching a predicate
nums.foldLeft(0)((count, x) => if (x % 2 == 0) count + 1 else count)

In [ ]:
// Implementing map using foldLeft
def myMap[A, B](list: List[A])(f: A => B): List[B] =
  list.foldLeft(List.empty[B])((acc, x) => acc :+ f(x))

// Implementing filter using foldLeft
def myFilter[A](list: List[A])(p: A => Boolean): List[A] =
  list.foldLeft(List.empty[A])((acc, x) => if (p(x)) acc :+ x else acc)

// Implementing flatMap using foldLeft
def myFlatMap[A, B](list: List[A])(f: A => List[B]): List[B] =
  list.foldLeft(List.empty[B])((acc, x) => acc ++ f(x))

myMap(List(1, 2, 3, 4))(_ * 10)
myFilter(List(1, 2, 3, 4, 5))(_ % 2 == 0)
myFlatMap(List(1, 2, 3))(n => List(n, n * 10))


### `foldLeft` for Aggregating Business Data


In [ ]:
case class Sale(product: String, quantity: Int, unitPrice: Double)

val sales = List(
  Sale("Widget", 10, 5.00),
  Sale("Gadget", 3,  25.00),
  Sale("Donut",  50, 1.50),
  Sale("Widget", 5,  5.00)
)

// Total revenue
val totalRevenue = sales.foldLeft(0.0)((acc, s) => acc + s.quantity * s.unitPrice)

// Group quantities by product using foldLeft
val quantityByProduct = sales.foldLeft(Map.empty[String, Int]) { (acc, s) =>
  acc.updated(s.product, acc.getOrElse(s.product, 0) + s.quantity)
}

totalRevenue
quantityByProduct


# 15. `foldRight` — Accumulate from Right to Left

`foldRight` is similar to `foldLeft` but processes elements from **right to left**.

Signature: `List[A].foldRight(z: B)(f: (A, B) => B): B`

Note: the argument order in `f` is reversed — the element comes **first**, the accumulator **second**.

`foldRight` is natural for building lists because it preserves element order without needing to reverse at the end.


In [ ]:
val nums = List(1, 2, 3, 4, 5)

// Build the list back using cons :: (natural for foldRight)
nums.foldRight(List.empty[Int])(_ :: _)

// Implementing map with foldRight (preserves order)
def mapViaFoldRight[A, B](list: List[A])(f: A => B): List[B] =
  list.foldRight(List.empty[B])(f(_) :: _)

mapViaFoldRight(List(1, 2, 3))(_ * 10)

// Difference: foldLeft builds right-to-left accumulation
nums.foldLeft(List.empty[Int])(_ :+ _)      // preserves order
nums.foldLeft(List.empty[Int])((acc, x) => x :: acc) // reverses


### `foldLeft` vs `foldRight` — Key Differences

| | `foldLeft` | `foldRight` |
|--|------------|-------------|
| Direction | Left to right | Right to left |
| Accumulator position | First arg of `f` | Second arg of `f` |
| Stack safety | Tail-recursive (safe) | May overflow on large lists |
| Natural for | Counters, maps, accumulators | Building lists, trees |
| Association | Left-associative: `((z op a) op b) op c` | Right-associative: `a op (b op (c op z))` |



# 16. `reduce` and `reduceLeft`

`reduce` is like `foldLeft` but **without an initial value** — it uses the first element as the starting accumulator.

This means `reduce` requires a **non-empty** collection — it throws an exception on an empty list.

Use `reduceOption` for safe versions that return `Option`.


In [ ]:
val nums = List(3, 1, 4, 1, 5, 9, 2, 6)

nums.reduce(_ + _)       // sum
nums.reduce(_ * _)       // product
nums.reduce(Math.max)    // max
nums.reduce(Math.min)    // min

// Safe version that handles empty lists
List.empty[Int].reduceOption(_ + _)   // None
nums.reduceOption(_ + _)              // Some(31)


# 17. `scan` — Running Accumulation

`scanLeft` is like `foldLeft` but returns **all intermediate accumulator values**, not just the final one.

It is extremely useful for computing **running totals**, **cumulative sums**, and **moving state**.

Signature: `List[A].scanLeft(z: B)(f: (B, A) => B): List[B]`

The result has one more element than the input — the first element is always the initial value `z`.


In [ ]:
val nums = List(1, 2, 3, 4, 5)

// Running sum
nums.scanLeft(0)(_ + _)
// Result: List(0, 1, 3, 6, 10, 15)

// Running product
nums.scanLeft(1)(_ * _)
// Result: List(1, 1, 2, 6, 24, 120)

// Running maximum
List(3, 1, 4, 1, 5, 9, 2, 6).scanLeft(Int.MinValue)(Math.max).tail

In [ ]:
// Real-world: running balance of a bank account
case class Transaction(description: String, amount: Double)

val transactions = List(
  Transaction("Opening deposit",  1000.00),
  Transaction("Rent payment",    -800.00),
  Transaction("Salary",          2500.00),
  Transaction("Groceries",        -120.00),
  Transaction("Electric bill",    -95.00)
)

val runningBalance = transactions.scanLeft(0.0)((balance, tx) => balance + tx.amount).tail

transactions.zip(runningBalance).foreach { case (tx, bal) =>
  println(f"${tx.description}%-20s ${tx.amount}%+10.2f  Balance: ${bal}%10.2f")
}


# 18. `groupBy`, `collect`, and `zip`

These three HOFs handle common but distinct patterns: grouping, conditional transformation, and parallel traversal.


In [ ]:
// groupBy: partition a collection into a Map keyed by a function
val words = List("apple", "banana", "cherry", "avocado", "blueberry", "almond")

val byFirstLetter = words.groupBy(_.head)
byFirstLetter

val byLength = words.groupBy(_.length)
byLength

In [ ]:
// collect: filter + map using a PartialFunction
val mixed: List[Any] = List(1, "hello", 2.5, 3, "world", 4, true)

// Extract only integers
val ints = mixed.collect { case n: Int => n }

// Extract strings and uppercase them
val upperStrings = mixed.collect { case s: String => s.toUpperCase }

ints
upperStrings

In [ ]:
// zip: combine two collections pairwise
val keys   = List("name", "age", "city")
val values = List("Alice", "30", "Istanbul")

val zipped = keys.zip(values)
zipped

// Convert to Map
zipped.toMap

// zipWithIndex: pair each element with its position
val fruits = List("apple", "banana", "cherry")
fruits.zipWithIndex
fruits.zipWithIndex.map { case (fruit, idx) => s"${idx + 1}. $fruit" }


# 19. `for`-Comprehensions — Syntactic Sugar for HOFs

Scala's `for`-comprehension is syntactic sugar that desugars into `flatMap`, `map`, and `filter` calls.

It provides a **sequential, readable syntax** for chaining these operations.

| `for`-comprehension syntax | Desugars to |
|---------------------------|-------------|
| `x <- collection` | `flatMap` (if more generators follow) or `map` |
| `if predicate` | `filter` or `withFilter` |
| `yield expr` | the body of the innermost `map` |


In [ ]:
// Simple for-comprehension
val nums = List(1, 2, 3, 4, 5)

val doubled = for {
  n <- nums
} yield n * 2

// Equivalent to: nums.map(n => n * 2)

// With filter
val evenDoubled = for {
  n <- nums
  if n % 2 == 0
} yield n * 2

// Equivalent to: nums.filter(_ % 2 == 0).map(_ * 2)

doubled
evenDoubled

In [ ]:
// Multiple generators — flatMap
val xs = List(1, 2, 3)
val ys = List(10, 20)

val pairs = for {
  x <- xs
  y <- ys
} yield (x, y)

// Equivalent to: xs.flatMap(x => ys.map(y => (x, y)))
pairs

In [ ]:
// for-comprehension over Option
def findUser(id: Int): Option[String]    = Map(1 -> "Alice", 2 -> "Bob").get(id)
def findEmail(name: String): Option[String] = Map("Alice" -> "alice@ex.com").get(name)
def findPhone(email: String): Option[String] = Map("alice@ex.com" -> "555-1234").get(email)

val phone = for {
  name  <- findUser(1)
  email <- findEmail(name)
  phone <- findPhone(email)
} yield phone

phone


The `for`-comprehension makes chained `flatMap` operations look like sequential imperative steps while remaining completely functional.



# 20. Closures — Definition and Core Concept

A **closure** is a function that **captures variables from its surrounding scope**.

When a function references a variable that is defined outside the function itself, it "closes over" that variable — hence the name **closure**.

The captured variable is called a **free variable** (as opposed to a **bound variable**, which is a parameter of the function itself).

Closures are central to functional programming because they allow functions to carry **context** without needing global state.


In [ ]:
// A simple closure — `factor` is a free variable captured from the enclosing scope
val factor = 3
val multiply: Int => Int = x => x * factor  // closes over `factor`

multiply(5)    // 15
multiply(10)   // 30

// The function `multiply` carries the value of `factor` with it
// even after the scope where `factor` was defined is gone

In [ ]:
// Closures capturing function parameters
def makeAdder(n: Int): Int => Int = {
  x => x + n   // closes over the parameter `n`
}

val add5  = makeAdder(5)
val add10 = makeAdder(10)
val add100 = makeAdder(100)

// Each function carries its own copy of `n`
add5(3)    // 8
add10(3)   // 13
add100(3)  // 103


Each call to `makeAdder` creates a **new closure** with its own captured `n`. The closures `add5`, `add10`, and `add100` are independent — modifying one does not affect the others.



# 21. Closures Capturing Mutable State

A closure can also capture a **mutable variable** (`var`). In that case, the function "sees" the variable's current value each time it is called.

This is generally discouraged in functional programming because it introduces hidden state, but it is important to understand how it works.


In [ ]:
// Closure over a mutable variable
var count = 0
val increment: () => Int = () => { count += 1; count }

increment()   // 1
increment()   // 2
increment()   // 3
count         // 3 — the outer variable has been mutated

In [ ]:
// Multiple closures sharing the same captured variable
var shared = 0
val incShared: () => Unit = () => { shared += 1 }
val decShared: () => Unit = () => { shared -= 1 }
val getShared: () => Int  = () => shared

incShared()
incShared()
incShared()
decShared()
getShared()   // 2


This pattern simulates an **object** using closures — the shared mutable variable is like a private field, and `incShared`, `decShared`, `getShared` are like methods. This is the functional alternative to a simple class.

In idiomatic Scala, however, we prefer **immutable closures** — those that only capture `val`s.



# 22. Closures as Configuration Carriers

One of the most useful practical patterns for closures is using them to carry **configuration or context** — eliminating the need to pass the same arguments repeatedly.


In [ ]:
// A database query executor that captures the connection string
def makeQueryRunner(connectionString: String): String => String = {
  query => s"[Executing on $connectionString]: $query"
}

val prodRunner = makeQueryRunner("jdbc:postgresql://prod-db:5432/app")
val devRunner  = makeQueryRunner("jdbc:postgresql://localhost:5432/dev")

prodRunner("SELECT * FROM users WHERE active = true")
devRunner("SELECT * FROM orders LIMIT 10")

In [ ]:
// A logger closure that captures log level and destination
def makeLogger(level: String, prefix: String): String => Unit = {
  message => println(s"[$level][$prefix] $message")
}

val infoLogger  = makeLogger("INFO",  "App")
val errorLogger = makeLogger("ERROR", "Database")
val warnLogger  = makeLogger("WARN",  "Auth")

infoLogger("Server started on port 8080")
errorLogger("Connection pool exhausted")
warnLogger("Invalid login attempt")


The loggers carry their configuration — level and prefix — through closures without needing global state or class instances.



# 23. Lexical Scope

**Scope** determines where a name (variable, function, type) is visible and accessible in a program.

Scala uses **lexical scope** (also called **static scope**): the scope of a name is determined by where it is **written** in the source code — not by where it is called at runtime.

This makes Scala code **predictable** — you can always determine a variable's scope by reading the source, without understanding the call stack.

The rules of lexical scope:
1. Inner scopes can access names from outer scopes
2. Outer scopes cannot access names from inner scopes
3. Inner names can **shadow** (hide) outer names with the same identifier


In [ ]:
val outerVal = 100

def outerFunction(): Unit = {
  val middleVal = 200   // visible inside outerFunction and all nested scopes

  def innerFunction(): Unit = {
    val innerVal = 300  // visible only inside innerFunction

    // Can see: outerVal, middleVal, innerVal
    println(s"Inner sees: $outerVal, $middleVal, $innerVal")
  }

  // Can see: outerVal, middleVal
  // Cannot see: innerVal (it's in a deeper scope)
  println(s"Middle sees: $outerVal, $middleVal")

  innerFunction()
}

outerFunction()


### Shadowing

When an inner scope declares a name that already exists in an outer scope, the inner name **shadows** the outer one.


In [ ]:
val x = 10

def demo(): Unit = {
  val x = 20   // shadows the outer x
  println(s"Inside demo: x = $x")   // 20

  {
    val x = 30   // shadows demo's x
    println(s"Inside block: x = $x") // 30
  }

  println(s"After block: x = $x")   // 20 (block's x is gone)
}

demo()
println(s"Outside: x = $x")   // 10 (original, untouched)


Shadowing is **legal** in Scala but should be used with care — it can make code harder to reason about. Most style guides recommend avoiding it unless the intent is clearly to override a name within a limited scope.



# 24. Block Scope

In Scala, every `{ }` block creates a **new scope**.

Variables declared inside a block are visible only within that block and its nested blocks.

Crucially, a block is also an **expression** — it evaluates to the value of its last statement. This allows complex initialization to be encapsulated cleanly.


In [ ]:
// Block as an expression
val result = {
  val a = 10
  val b = 20
  val c = a + b     // c is local to this block
  c * 2             // the block evaluates to this value
}

// c is not accessible here — it went out of scope
result   // 60

In [ ]:
// Practical use: complex initialization without polluting the outer scope
val config = {
  val rawHost = "  localhost  "
  val rawPort = "8080"
  val cleanHost = rawHost.trim
  val cleanPort = rawPort.trim.toInt
  (cleanHost, cleanPort)  // only this tuple escapes the block
}

// rawHost, rawPort, cleanHost, cleanPort are all out of scope here
config

In [ ]:
// if-else and match are also expressions in Scala
val age = 25

val category = if (age < 18) "Minor"
               else if (age < 65) "Adult"
               else "Senior"

val label = age match {
  case a if a < 18 => "Minor"
  case a if a < 65 => "Adult"
  case _           => "Senior"
}

category
label


# 25. Closure Scope — How Closures Capture Their Environment

When a closure is created, it captures **a reference to** its free variables (not a copy of their value at creation time for `var`, but literally the variable itself).

For **immutable values** (`val`), this distinction doesn't matter — the value cannot change.

For **mutable variables** (`var`), the closure captures the variable itself — so it always sees the current value.

Understanding this distinction is key to avoiding subtle bugs.


In [ ]:
// Closure captures a val — captured at the moment the closure is created
val closures = (1 to 5).map { i =>
  val captured = i   // immutable — captured by value semantics
  () => captured
}

closures.map(f => f())   // List(1, 2, 3, 4, 5) — each closure has its own `captured`

In [ ]:
// Closure captures a var — shares the variable
var message = "initial"
val readMessage: () => String = () => message   // closes over the var

readMessage()      // "initial"

message = "updated"
readMessage()      // "updated" — sees the new value

message = "final"
readMessage()      // "final"


### Scope Chain and Nested Closures

Closures can be nested — an inner closure can capture variables from both the immediate enclosing function and outer functions.


In [ ]:
def outerFn(a: Int): Int => Int => Int = {
  b: Int => {
    c: Int => a + b + c   // innermost closure captures a, b, c from 3 different scopes
  }
}

val step1 = outerFn(10)     // captures a=10
val step2 = step1(20)       // captures b=20
val result = step2(30)      // captures c=30

result   // 10 + 20 + 30 = 60


This is actually **currying implemented via closures** — each application returns a new closure that carries one more piece of captured state.



# 26. Scope in `for`-Comprehensions

Within a `for`-comprehension, each generator introduces names into scope for all subsequent generators and the `yield` expression. This is called **sequential scoping**.


In [ ]:
// Each name introduced by <- is in scope for all subsequent lines
val result = for {
  x <- List(1, 2, 3)           // x is in scope from here
  y <- List(x * 10, x * 100)   // y is in scope from here; x is still in scope
  if y > 20                    // both x and y are in scope
  z = x + y                    // local val: z is in scope from here
} yield s"x=$x y=$y z=$z"

result


This sequential scoping of `for`-comprehensions means that later generators can reference bindings from earlier ones — unlike parallel comprehensions in some other languages.



# 27. Closures in Collection Pipelines

In practice, closures are used constantly in collection pipelines — the lambdas passed to `map`, `filter`, `foldLeft` etc. are all closures when they reference external variables.


In [ ]:
// The lambda captures `threshold` from the outer scope
val threshold = 50
val scores = List(45, 62, 30, 78, 50, 91, 22)

val passing = scores.filter(_ >= threshold)  // lambda closes over `threshold`
val scaled  = scores.map(s => (s.toDouble / threshold) * 100)  // closes over `threshold`

passing
scaled

In [ ]:
// A more complex example: closures capturing a configuration map
val exchangeRates = Map("USD" -> 1.0, "EUR" -> 0.92, "GBP" -> 0.79, "TRY" -> 32.5)

case class Price(amount: Double, currency: String)

// This lambda closes over `exchangeRates`
val convertToUsd: Price => Double = price =>
  price.amount / exchangeRates.getOrElse(price.currency, 1.0)

val prices = List(
  Price(100.0, "EUR"),
  Price(50.0,  "GBP"),
  Price(1000.0, "TRY")
)

prices.map(convertToUsd)


The `convertToUsd` function is a closure — it carries the `exchangeRates` map with it. When passed to `map`, it uses that captured map for each conversion without needing to accept it as a parameter.



# 28. Combining All Techniques — Real-World Order Processing Pipeline

Now we will bring every concept from this lecture together in a realistic **order processing system**.

This scenario demonstrates:
- ADTs for domain modeling (orders, line items, statuses)
- Pattern matching for state transitions and business rules
- HOFs (`map`, `filter`, `flatMap`, `foldLeft`, `groupBy`) for data processing
- Closures for carrying configuration (discount rates, tax rates)
- Scoping to keep intermediate values contained


In [ ]:
// =========================================
// Domain Model — ADTs
// =========================================

sealed trait OrderStatus
case object Draft     extends OrderStatus
case object Submitted extends OrderStatus
case object Approved  extends OrderStatus
case object Shipped   extends OrderStatus
case class  Rejected(reason: String) extends OrderStatus

sealed trait CustomerTier
case object Bronze  extends CustomerTier
case object Silver  extends CustomerTier
case object Gold    extends CustomerTier
case object Platinum extends CustomerTier

case class LineItem(productId: String, name: String, qty: Int, unitPrice: Double)
case class Customer(id: String, name: String, tier: CustomerTier)
case class Order(id: String, customer: Customer, items: List[LineItem], status: OrderStatus)

In [ ]:
// =========================================
// Business Rules — Closures carrying config
// =========================================

// Discount rates by tier — captured by closures below
val discountRates: Map[CustomerTier, Double] = Map(
  Bronze   -> 0.00,
  Silver   -> 0.05,
  Gold     -> 0.10,
  Platinum -> 0.15
)

val taxRate = 0.18  // VAT rate — captured by closures

// Closures that capture the configuration maps above
val discountFor: CustomerTier => Double =
  tier => discountRates.getOrElse(tier, 0.0)

val lineItemTotal: LineItem => Double =
  item => item.qty * item.unitPrice

val applyDiscount: CustomerTier => Double => Double =
  tier => subtotal => subtotal * (1.0 - discountFor(tier))

val applyTax: Double => Double =
  subtotal => subtotal * (1.0 + taxRate)  // closes over taxRate

In [ ]:
// =========================================
// Order Validation — Pattern Matching + ADTs
// =========================================

def validateOrder(order: Order): Order = {
  val issues = {
    val emptyCheck   = if (order.items.isEmpty) List("Order has no items") else Nil
    val negQtyCheck  = order.items.collect {
      case LineItem(_, name, qty, _) if qty <= 0 => s"Invalid quantity for $name"
    }
    val negPriceCheck = order.items.collect {
      case LineItem(_, name, _, price) if price <= 0 => s"Invalid price for $name"
    }
    emptyCheck ++ negQtyCheck ++ negPriceCheck
  }

  issues match {
    case Nil     => order.copy(status = Approved)
    case reasons => order.copy(status = Rejected(reasons.mkString("; ")))
  }
}

In [ ]:
// =========================================
// Order Pricing — HOFs + Closures
// =========================================

case class PricedOrder(order: Order, subtotal: Double, discount: Double, tax: Double, total: Double)

def priceOrder(order: Order): Option[PricedOrder] = order.status match {
  case Approved =>
    val subtotal   = order.items.foldLeft(0.0)((acc, item) => acc + lineItemTotal(item))
    val discounted = applyDiscount(order.customer.tier)(subtotal)
    val discountAmt = subtotal - discounted
    val total       = applyTax(discounted)
    val taxAmt      = total - discounted
    Some(PricedOrder(order, subtotal, discountAmt, taxAmt, total))
  case _ => None
}

In [ ]:
// =========================================
// Sample Data
// =========================================

val orders = List(
  Order("ORD-001",
    Customer("C1", "Alice", Gold),
    List(
      LineItem("P01", "Laptop",  1, 999.99),
      LineItem("P02", "Mouse",   2, 29.99)
    ),
    Submitted
  ),
  Order("ORD-002",
    Customer("C2", "Bob", Bronze),
    List(
      LineItem("P03", "Keyboard", 1, 79.99)
    ),
    Submitted
  ),
  Order("ORD-003",
    Customer("C3", "Charlie", Platinum),
    List(
      LineItem("P04", "Monitor",  2, 349.00),
      LineItem("P05", "Webcam",   1, -10.0)   // invalid price!
    ),
    Submitted
  ),
  Order("ORD-004",
    Customer("C4", "Diana", Silver),
    List(),  // empty order — should be rejected
    Submitted
  )
)

In [ ]:
// =========================================
// Execute the Pipeline
// =========================================

val validatedOrders = orders.map(validateOrder)
val pricedOrders    = validatedOrders.flatMap(priceOrder)

// Print results
println("=== Pricing Results ===")
pricedOrders.foreach { po =>
  println(f"${po.order.id}  ${po.order.customer.name}%-10s  " +
          f"Subtotal: $$${po.subtotal}%8.2f  " +
          f"Discount: $$${po.discount}%6.2f  " +
          f"Tax: $$${po.tax}%6.2f  " +
          f"Total: $$${po.total}%8.2f")
}

// Print rejected orders
println("\n=== Rejected Orders ===")
validatedOrders.collect {
  case o @ Order(id, customer, _, Rejected(reason)) =>
    println(s"$id (${customer.name}): $reason")
}

In [ ]:
// =========================================
// Analytics — groupBy, foldLeft, scan
// =========================================

// Revenue by customer tier
val revenueByTier = pricedOrders
  .groupBy(_.order.customer.tier)
  .map { case (tier, orders) =>
    tier -> orders.foldLeft(0.0)((acc, o) => acc + o.total)
  }

println("\n=== Revenue by Tier ===")
revenueByTier.foreach { case (tier, revenue) =>
  println(f"$tier: $$${revenue}%.2f")
}

// Running total of revenue
val runningRevenue = pricedOrders
  .map(_.total)
  .scanLeft(0.0)(_ + _)
  .tail

println("\n=== Running Revenue ===")
pricedOrders.zip(runningRevenue).foreach { case (po, running) =>
  println(f"After ${po.order.id}: cumulative = $$${running}%.2f")
}


This pipeline demonstrates every technique from this lecture working together:

| Technique | Where used |
|-----------|------------|
| Sum types (sealed trait) | `OrderStatus`, `CustomerTier` |
| Product types (case class) | `LineItem`, `Customer`, `Order`, `PricedOrder` |
| Recursive ADTs | `Order` contains `List[LineItem]` |
| Pattern matching | `validateOrder`, `priceOrder`, analytics |
| Guards in patterns | Validation checks (`if qty <= 0`) |
| Nested patterns | `Order(id, customer, _, Rejected(reason))` |
| `map` | Apply validation, apply pricing, extract fields |
| `filter` | Implicit via `collect` and `flatMap` |
| `flatMap` | Chain optional pricing, split sentences |
| `foldLeft` | Sum revenue, build maps |
| `collect` | Extract rejected orders, validate |
| `groupBy` | Revenue by tier |
| `scanLeft` | Running revenue |
| `zip` | Pair orders with running totals |
| Closures | `discountFor`, `applyDiscount`, `applyTax` capturing config |
| Lexical scope | All intermediate vals are block-scoped |
| Block expressions | Validation logic in `{ }` returning a list |



# Final Summary

### Algebraic Data Types

| Concept | Scala Syntax | Key Property |
|---------|-------------|-------------|
| Product type | `case class Foo(a: A, b: B)` | Holds all fields simultaneously |
| Sum type | `sealed trait + case class/object` | Represents one of the alternatives |
| Recursive ADT | Sum + Product referencing itself | Models trees, lists, expressions |
| `sealed` | Restricts subclasses to same file | Enables exhaustiveness checking |

### Pattern Matching

| Feature | Syntax | Use Case |
|---------|--------|----------|
| Literal match | `case 42 =>` | Dispatch on specific values |
| Wildcard | `case _ =>` | Default/catch-all case |
| Destructuring | `case Foo(a, b) =>` | Extract fields from case class |
| Guard | `case x if x > 0 =>` | Conditional match |
| Binding | `case f @ Foo(a) =>` | Bind whole value and destructure |
| Type pattern | `case s: String =>` | Dispatch on runtime type |
| List pattern | `case h :: t =>` | Destructure a list |
| Nested | `case Foo(Bar(x)) =>` | Deep destructuring |

### Higher-Order Functions on Collections

| Function | Signature | Use Case |
|----------|-----------|----------|
| `map` | `(A => B) => List[B]` | Transform every element |
| `filter` | `(A => Boolean) => List[A]` | Keep matching elements |
| `flatMap` | `(A => List[B]) => List[B]` | Transform and flatten |
| `foldLeft` | `(B, (B,A) => B) => B` | Accumulate left to right |
| `foldRight` | `(B, (A,B) => B) => B` | Accumulate right to left |
| `reduce` | `((A,A) => A) => A` | Combine all elements |
| `scanLeft` | `(B, (B,A) => B) => List[B]` | Running accumulation |
| `groupBy` | `(A => K) => Map[K, List[A]]` | Partition by key |
| `collect` | `PartialFunction[A, B] => List[B]` | Filter + transform |
| `zip` | `List[B] => List[(A,B)]` | Pair with another list |
| `partition` | `(A => Boolean) => (List[A], List[A])` | Split in two |

### Closures and Scopes

| Concept | Description |
|---------|-------------|
| Closure | Function that captures variables from its defining scope |
| Free variable | Variable in a closure not defined as its parameter |
| Lexical scope | Scope determined by source position, not call stack |
| Block scope | Every `{ }` creates a new scope; block is an expression |
| Shadowing | Inner declaration hides outer with the same name |
| Closure scope | Each closure instance has its own captured environment |

### Principles These Techniques Promote

1. **Correctness** — sealed ADTs make illegal states unrepresentable
2. **Exhaustiveness** — pattern matching on sealed traits is compiler-checked
3. **Expressiveness** — HOFs replace verbose loops with declarative pipelines
4. **Modularity** — closures carry configuration cleanly without global state
5. **Immutability** — case classes and `val` closures prevent hidden mutation bugs
6. **Composability** — HOFs chain naturally into readable, layered pipelines

Mastering these techniques is essential for working effectively in **Scala, Apache Spark, Cats, ZIO, Akka Streams**, and any modern functional Scala codebase.
